# x64 Secondary Holdout 70k 3p5k Build

This notebook regenerates the secondary `70k` normal and `3.5k` defect holdout used for larger-scale evaluation.

Unlike the main benchmark notebook, this branch preserves the original `50k 5%` train and validation rows and only replaces the test split with a larger disjoint holdout.


## Notebook Flow

Run the notebook from top to bottom.

1. load the holdout config for this branch
2. confirm the raw pickle and base `50k 5%` metadata exist
3. run the shared secondary-holdout builder
4. validate the generated holdout metadata and arrays
5. inspect the resulting split summary and defect breakdown


In [ ]:
try:
    import os
    import subprocess
    import sys
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from IPython.display import display
    REPO_ROOT = Path.cwd().resolve()
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            REPO_ROOT = candidate
            break
    else:
        raise FileNotFoundError('Could not locate the repository root.')
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    if str(REPO_ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / 'src'))
    DATASET_DIR = REPO_ROOT / 'data' / 'dataset' / 'x64' / 'holdout70k_3p5k'
    CONFIG_PATH = DATASET_DIR / 'data_config.toml'

    def repo_path(path_like: str | Path) -> Path:
        path = Path(path_like)
        return path if path.is_absolute() else REPO_ROOT / path
    (REPO_ROOT, CONFIG_PATH)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import os\nimport subprocess\nimport sys\nfrom pathlib import path\nimport numpy as np\nimport pandas as pd\nfrom ipython.display import display\nrepo_root = path.cwd().resolve()\nfor candidate in [repo_root, *repo_root.parents]:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        repo_root = candidate\n        break\nelse:\n    raise filenotfounderror('could not locate the repository root.')\nif str(repo_root) not in sys.path:\n    sys.path.insert(0, str(repo_root))\nif str(repo_root / 'src') not in sys.path:\n    sys.path.insert(0, str(repo_root / 'src'))\ndataset_dir = repo_root / 'data' / 'dataset' / 'x64' / 'holdout70k_3p5k'\nconfig_path = dataset_dir / 'data_config.toml'\n\ndef repo_path(path_like: str | path) -> path:\n    path = path(path_like)\n    return path if path.is_absolute() else repo_root / path\n(repo_root, config_path)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
RUN_TRAINING = False
print(f'RUN_TRAINING = {RUN_TRAINING}')


In [ ]:
from wafer_defect.config import load_toml
config = load_toml(CONFIG_PATH)
holdout_cfg = config['holdout']
raw_pickle = repo_path(holdout_cfg['raw_pickle'])
base_metadata_path = repo_path(holdout_cfg['base_metadata_csv'])
output_metadata_path = repo_path(holdout_cfg['output_metadata_csv'])
output_arrays_dir = output_metadata_path.parent / f"arrays_{output_metadata_path.stem.removeprefix('metadata_')}"
config_summary = pd.DataFrame([{'field': 'raw_pickle', 'value': holdout_cfg['raw_pickle']}, {'field': 'base_metadata_csv', 'value': holdout_cfg['base_metadata_csv']}, {'field': 'output_metadata_csv', 'value': holdout_cfg['output_metadata_csv']}, {'field': 'image_size', 'value': holdout_cfg['image_size']}, {'field': 'test_normal_count', 'value': holdout_cfg['test_normal_count']}, {'field': 'test_defect_count', 'value': holdout_cfg['test_defect_count']}, {'field': 'random_seed', 'value': holdout_cfg['random_seed']}, {'field': 'expected_arrays_dir', 'value': output_arrays_dir.relative_to(REPO_ROOT).as_posix()}])
display(config_summary)


## Input Checks

The holdout builder depends on two existing inputs:

- the raw WM-811K pickle
- the previously generated base metadata file `metadata_50k_5pct.csv`


In [ ]:
try:
    if not raw_pickle.exists():
        raise FileNotFoundError(f'Raw dataset file not found: {raw_pickle}')
    if not base_metadata_path.exists():
        raise FileNotFoundError(f'Base metadata file not found: {base_metadata_path}')
    base_metadata = pd.read_csv(base_metadata_path)
    display(base_metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if not raw_pickle.exists():\n    raise filenotfounderror(f'raw dataset file not found: {raw_pickle}')\nif not base_metadata_path.exists():\n    raise filenotfounderror(f'base metadata file not found: {base_metadata_path}')\nbase_metadata = pd.read_csv(base_metadata_path)\ndisplay(base_metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Generate the Secondary Holdout

This cell runs the dedicated holdout builder script and writes the new metadata and arrays into `data/processed/x64/wm811k/`.


In [ ]:
try:
    command = [sys.executable, 'scripts/build_secondary_holdout_metadata.py', '--raw-pickle', str(Path(holdout_cfg['raw_pickle'])), '--base-metadata', str(Path(holdout_cfg['base_metadata_csv'])), '--output-metadata', str(Path(holdout_cfg['output_metadata_csv'])), '--test-normal-count', str(holdout_cfg['test_normal_count']), '--test-defect-count', str(holdout_cfg['test_defect_count']), '--image-size', str(holdout_cfg['image_size']), '--seed', str(holdout_cfg['random_seed'])]
    pythonpath_entries = [str(REPO_ROOT), str(REPO_ROOT / 'src')]
    existing_pythonpath = os.environ.get('PYTHONPATH')
    if existing_pythonpath:
        pythonpath_entries.append(existing_pythonpath)
    run_env = os.environ.copy()
    run_env['PYTHONPATH'] = os.pathsep.join(pythonpath_entries)
    result = subprocess.run(command, cwd=REPO_ROOT, check=True, capture_output=True, text=True, env=run_env)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "command = [sys.executable, 'scripts/build_secondary_holdout_metadata.py', '--raw-pickle', str(path(holdout_cfg['raw_pickle'])), '--base-metadata', str(path(holdout_cfg['base_metadata_csv'])), '--output-metadata', str(path(holdout_cfg['output_metadata_csv'])), '--test-normal-count', str(holdout_cfg['test_normal_count']), '--test-defect-count', str(holdout_cfg['test_defect_count']), '--image-size', str(holdout_cfg['image_size']), '--seed', str(holdout_cfg['random_seed'])]\npythonpath_entries = [str(repo_root), str(repo_root / 'src')]\nexisting_pythonpath = os.environ.get('pythonpath')\nif existing_pythonpath:\n    pythonpath_entries.append(existing_pythonpath)\nrun_env = os.environ.copy()\nrun_env['pythonpath'] = os.pathsep.join(pythonpath_entries)\nresult = subprocess.run(command, cwd=repo_root, check=true, capture_output=true, text=true, env=run_env)\nprint(result.stdout)\nif result.stderr.strip():\n    print(result.stderr)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Validate the Written Holdout Outputs


In [ ]:
try:
    if not output_metadata_path.exists():
        raise FileNotFoundError(f'Expected holdout metadata file was not created: {output_metadata_path}')
    if not output_arrays_dir.exists():
        raise FileNotFoundError(f'Expected holdout arrays directory was not created: {output_arrays_dir}')
    holdout_metadata = pd.read_csv(output_metadata_path)
    display(holdout_metadata.head())
    display(holdout_metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))
    array_files = sorted(output_arrays_dir.glob('*.npy'))
    array_file_names = {path.name for path in array_files}
    test_rows = holdout_metadata[holdout_metadata['split'] == 'test'].copy()
    metadata_file_names = set(test_rows['array_path'].map(lambda value: Path(value).name))
    missing_arrays = sorted(metadata_file_names - array_file_names)
    extra_arrays = sorted(array_file_names - metadata_file_names)
    validation_df = pd.DataFrame([{'total_metadata_rows': len(holdout_metadata), 'test_rows': len(test_rows), 'array_files': len(array_files), 'missing_arrays': len(missing_arrays), 'extra_arrays': len(extra_arrays)}])
    display(validation_df)
    assert len(missing_arrays) == 0, f'Metadata references missing arrays, e.g. {missing_arrays[:5]}'
    assert len(extra_arrays) == 0, f'Arrays directory contains unexpected extra files, e.g. {extra_arrays[:5]}'
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if not output_metadata_path.exists():\n    raise filenotfounderror(f'expected holdout metadata file was not created: {output_metadata_path}')\nif not output_arrays_dir.exists():\n    raise filenotfounderror(f'expected holdout arrays directory was not created: {output_arrays_dir}')\nholdout_metadata = pd.read_csv(output_metadata_path)\ndisplay(holdout_metadata.head())\ndisplay(holdout_metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))\narray_files = sorted(output_arrays_dir.glob('*.npy'))\narray_file_names = {path.name for path in array_files}\ntest_rows = holdout_metadata[holdout_metadata['split'] == 'test'].copy()\nmetadata_file_names = set(test_rows['array_path'].map(lambda value: path(value).name))\nmissing_arrays = sorted(metadata_file_names - array_file_names)\nextra_arrays = sorted(array_file_names - metadata_file_names)\nvalidation_df = pd.dataframe([{'total_metadata_rows': len(holdout_metadata), 'test_rows': len(test_rows), 'array_files': len(array_files), 'missing_arrays': len(missing_arrays), 'extra_arrays': len(extra_arrays)}])\ndisplay(validation_df)\nassert len(missing_arrays) == 0, f'metadata references missing arrays, e.g. {missing_arrays[:5]}'\nassert len(extra_arrays) == 0, f'arrays directory contains unexpected extra files, e.g. {extra_arrays[:5]}'\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    holdout_test = holdout_metadata[holdout_metadata['split'] == 'test'].copy()
    display(holdout_test[holdout_test['is_anomaly'] == 1]['defect_type'].value_counts().rename_axis('defect_type').to_frame('count'))
    sample_rows = holdout_test.sample(n=min(6, len(holdout_test)), random_state=int(holdout_cfg['random_seed']))
    sample_shapes = []
    for _, row in sample_rows.iterrows():
        wafer = np.load(repo_path(row['array_path']))
        sample_shapes.append({'array_path': row['array_path'], 'shape': tuple(wafer.shape), 'min': float(wafer.min()), 'max': float(wafer.max())})
    display(pd.DataFrame(sample_shapes))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "holdout_test = holdout_metadata[holdout_metadata['split'] == 'test'].copy()\ndisplay(holdout_test[holdout_test['is_anomaly'] == 1]['defect_type'].value_counts().rename_axis('defect_type').to_frame('count'))\nsample_rows = holdout_test.sample(n=min(6, len(holdout_test)), random_state=int(holdout_cfg['random_seed']))\nsample_shapes = []\nfor _, row in sample_rows.iterrows():\n    wafer = np.load(repo_path(row['array_path']))\n    sample_shapes.append({'array_path': row['array_path'], 'shape': tuple(wafer.shape), 'min': float(wafer.min()), 'max': float(wafer.max())})\ndisplay(pd.dataframe(sample_shapes))\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Why This Notebook Exists

The holdout protocol is a separate dataset branch, not just a different evaluation command. Giving it its own notebook makes the train/val-preserving split logic reproducible for graders and keeps it separate from the main benchmark generation notebook.
